# 04 Distillation with a control
Goal: measure how much of the ensemble's gain a small student keeps, against a control. The teacher is the average of the tuned LightGBM and a random forest. The code that runs all seeds will live in `src/distill.py`.

## 1. Prediction (written before running anything)

**What is measured.** Each student is trained three ways: on true labels (the control, arm 1), on the teacher's out-of-fold labels for the same rows (arm 2), and on teacher labels for the real rows plus synthetic rows (arm 3). For a student and arm:

> share of the teacher's gain kept = (control RMSE − student RMSE) / (control RMSE − teacher RMSE)

0% means no better than the control, 100% means as good as the teacher, and a negative share means worse than the control.

**Expected shares** (test RMSE, mean over 5 seeds):

| Student | Arm 2: teacher labels, same rows | Arm 3: teacher labels + synthetic rows |
|---|---|---|
| MLP, 2 layers of 32 | about 10% | about 40% |
| MLP, 2 layers of 128 | about 15% | about 50% |
| Depth-limited tree | about 15% | about 35% |

**Why.**
- **Arm 2 helps only a little.** The student sees the same inputs as the control. The teacher's labels are smoother than the true ones, which removes some noise, but nothing new is learned about where the data lies.
- **Arm 3 helps most.** The synthetic rows give the student many more places to copy the teacher's function, and a function is easier to copy than noisy labels. This is where I expect the real gain.
- **The bigger MLP keeps more** because it has the capacity to use the extra rows. The small MLP runs out of capacity first.
- **The tree has the most to gain but the lowest ceiling.** Its control is the weakest, so the teacher's gain over it is large. But a depth-limited tree can only draw a blocky copy of the teacher, so it cannot keep most of that gain.

**What would surprise me.**
- Arm 2 beating arm 3.
- A negative share for any arm, meaning distillation hurt.
- The tree keeping more than either MLP.

A null result still counts. If distillation does not beat the control by more than the seed-to-seed noise, that is what gets reported.